# 從零開始做！超簡陋 RAG :)
這邊提供的步驟都是超級省略版，目的導向。  
如果想找完整的是錯過成家超醜但詳細解釋的話，請見 (這裡)[/1004-LLM/exp/aqing]  

為什麼中英夾雜因為老子有時候真的懶得切鍵盤，然後寫英文就是為了裝逼      
為什麼英文的語法像大便因為老子的母語就不是英文  

## Outline
- Set the Environment
- Embedding Model for Single Sentence
    * Tokenizer 
    * Embedding Model (make token embeddings)
    * 池化
    * (單句) 語意比較
- 打包手作嵌入模型: qingEmbedding()
- 分割大篇文本
- 儲存資料

## Set the Environment
### Hardware
All code in this notebook was tested on MacBook Air without 獨顯.  
The runtime is lightweight (if you use the same LM as I did.) and each cell should finish quickly (usually < 1 min),  
so 電腦的配置似乎並不是非常重要 for this tutorial.
### Internet
The function `transformers.AutoModel.from_pretrained()` will automatically download the required model from Hugging Face.   
Therefore, you need an internet connection when running the code for the first time.  
### Code env
This tutorial is based on **Python**. Required packages are listed below. You can also find detail in `reauirements.txt`.  
It is recommended to install them in a virtual environment like `conda` or something else, using either `pip install` or `conda install`.  
**requirements**
- pathlib
- pytorch (torch)
- time
- transformers

In [ ]:
### --------------------  Import python Models -------------------- ###
from pathlib import Path
import time
import torch
from transformers import AutoTokenizer, AutoModel

timeStart = time.time()

### ----------------------------  Text ---------------------------- ###
root = Path(__file__).resolve().parents[0]
textPath = f'{root}/In-the-Second-Beginning.txt'
textFile = open(textPath, 'r') # Read-only
textFile.close()

testString_1 = "I like astronomy."
testString_2 = "I enjoy watching the night sky full of stars."
testString_3 = "I like tomatoes."
testString_4 = 'I trust the universe will always bring me to you'

## Embedding Model for Single Sentence


### Tokenizer
Tokenizer 中文叫分詞器。可以把自然語言句子先分割成 token, 再轉成 tokenID.  
分割的規則取決于使用的語言模型，英文可能會有拆字跟的情況，比如複數的 's' 自己算一個 token.  
轉成 tokenID 的方法是查字典，字典(voca)也是語言模型自帶的。  

好欸什麼都用別人的，就這個開源爽。

In [ ]:
### --------------------------  Tokenizer ------------------------- ###
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
inp_1 = tokenizer(testString_1, return_tensors='pt')
inp_2 = tokenizer(testString_2, return_tensors='pt')
inp_3 = tokenizer(testString_3, return_tensors='pt') 
inp_4 = tokenizer(testString_4, return_tensors='pt')
re_tokens = tokenizer.convert_ids_to_tokens(inp_4['input_ids'][0]) # tokenID 還原成字, 超樸實的函數命名


#### 一點解釋
`AutoTokenizer.from_pretrained()` 會去 huggingFace 找到並**下載**語言模型，因為 `transformers` 就是 HF 發行的套件，所以記得聯網。  

這邊使用的語言模型叫做 `all-MiniLM-L6-v2"`，是 HF 上語意判斷領域裡面最多人下載的一款模型。  
當然也可以用別的你喜歡的，但就不保證運行的需求和時間ㄌ  

`inp_*` 是將 `teatString_*` 放進分詞器分割的產物，本體是一個類似字典的資料結構。裡面包含 `'input_ids'`（就是 tokenID 本人！） 和 `'attention_mask'`，  
這兩個是有用的等下還要用。

`tokenizer(return_tensors='pt')` 的 `'pt'` 代表 pytorch，接下來的張量運算會用到火炬蟒，所以把 tensor 打包成 pytorch 能認的格式。也可以 `='tf'` for tensorflow.  

### Embedding Model (make token embeddings)
將 tokenIDs

In [ ]:
### -----------------------  Token Embeddings ----------------------- ###
ebModel = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2') # 和寫 tokenizer 一樣的邏輯
torch.set_grad_enabled(False) # 不知道為什麼但先關掉梯度
out_1 = ebModel(**inp_1) # **代表全部帶入, inp 裡面有 'input_ids', 'token_type_ids' ...
out_2 = ebModel(**inp_2)
out_3 = ebModel(**inp_3)
torch.set_grad_enabled(True) # 喔喔, 訓練模型(改變)的時候才需要梯度, 只是使用模型的話不用, 所以關掉省電
                             # 開著也沒關係啦但我要善待 feifei